# POC Detector Shelf Detection di Google Colab

Notebook ini membuktikan bahwa `best.pt` hasil training dapat mendeteksi produk pada foto rak. POC mencakup inferensi satu gambar, visualisasi bounding box, ringkasan confidence dan latency, penyimpanan JSON, serta evaluasi test set secara opsional.

POC memuat model langsung dengan Ultralytics. Cara ini sengaja tidak menggunakan wrapper ONNX atau compliance engine repo, sehingga hasil detector dapat dinilai secara terpisah.

## 1. Konfigurasi POC

Nilai yang paling sering diubah diletakkan di satu cell:

- `MODEL_PATH`: model yang dihasilkan notebook training.
- `INPUT_MODE`: gunakan `upload` untuk memilih gambar dari komputer atau `drive` untuk membaca gambar di Drive.
- `CONFIDENCE`: batas minimum confidence detection.
- `RUN_DATASET_EVALUATION`: aktifkan bila ingin menghitung mAP, precision, dan recall menggunakan test set SKU-110K resmi dari Ultralytics.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone

DRIVE_ROOT = Path("/content/drive/MyDrive/shelf-detection")
MODEL_PATH = DRIVE_ROOT / "models/best.pt"
OUTPUT_DIR = DRIVE_ROOT / "poc_outputs"
DATASET_NAME = "SKU-110K.yaml"

INPUT_MODE = "upload"  # Pilihan: "upload" atau "drive"
DRIVE_IMAGE_PATH = DRIVE_ROOT / "poc_images/shelf.jpg"
CONFIDENCE = 0.25
IOU_THRESHOLD = 0.45
IMAGE_SIZE = 640
RUN_DATASET_EVALUATION = False

if INPUT_MODE not in {"upload", "drive"}:
    raise ValueError("INPUT_MODE harus 'upload' atau 'drive'.")

print(f"Model      : {MODEL_PATH}")
print(f"Input mode : {INPUT_MODE}")
print(f"Confidence : {CONFIDENCE}")

## 2. Mount Google Drive

Google Drive berisi `best.pt` dan menjadi lokasi permanen untuk gambar anotasi serta laporan JSON. Cell juga memastikan model memang tersedia sebelum dependency dan gambar diproses.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Model tidak ditemukan: {MODEL_PATH}. Jalankan notebook training terlebih dahulu."
    )

model_sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
print(f"Model ditemukan ({MODEL_PATH.stat().st_size / 1024**2:.1f} MB)")
print(f"SHA-256: {model_sha256}")

## 3. Instal dependency dan pilih device

POC hanya memerlukan Ultralytics, OpenCV headless, dan Matplotlib. GPU digunakan jika tersedia; POC tetap dapat berjalan di CPU untuk satu gambar, meskipun latency-nya lebih tinggi.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics>=8.4.70",
        "opencv-python-headless>=4.8",
        "matplotlib>=3.7",
    ],
    check=True,
)

import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 4. Pilih foto rak

Pada mode `upload`, Colab membuka pemilih file dari komputer. Pada mode `drive`, notebook menggunakan `DRIVE_IMAGE_PATH`. Format yang diterima dibatasi ke JPG, JPEG, PNG, dan WEBP.

In [ ]:
ALLOWED_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}

if INPUT_MODE == "upload":
    from google.colab import files

    uploaded = files.upload()
    uploaded_images = [
        Path(name) for name in uploaded if Path(name).suffix.lower() in ALLOWED_IMAGE_SUFFIXES
    ]
    if not uploaded_images:
        raise RuntimeError("Tidak ada file gambar yang valid pada hasil upload.")
    image_path = uploaded_images[0]
else:
    image_path = DRIVE_IMAGE_PATH

if not image_path.is_file():
    raise FileNotFoundError(f"Gambar tidak ditemukan: {image_path}")
if image_path.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES:
    raise ValueError(f"Format gambar tidak didukung: {image_path.suffix}")

image = cv2.imread(str(image_path))
if image is None:
    raise RuntimeError(f"OpenCV gagal membaca gambar: {image_path}")

height, width = image.shape[:2]
print(f"Gambar: {image_path} ({width}x{height})")

## 5. Muat model dan jalankan inferensi

`YOLO(MODEL_PATH)` memuat checkpoint PyTorch. `model.predict()` melakukan preprocessing, inferensi, non-maximum suppression, dan mengembalikan bounding box.

- `conf` membuang detection dengan confidence rendah.
- `iou` mengendalikan penggabungan box yang saling bertumpuk.
- `imgsz` menentukan resolusi input model.
- `wall_clock_ms` mengukur waktu total pemanggilan dari notebook.

In [ ]:
model = YOLO(str(MODEL_PATH))

started = time.perf_counter()
results = model.predict(
    source=str(image_path),
    conf=CONFIDENCE,
    iou=IOU_THRESHOLD,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    verbose=False,
)
wall_clock_ms = (time.perf_counter() - started) * 1000

if not results:
    raise RuntimeError("Ultralytics tidak mengembalikan hasil inferensi.")

result = results[0]
detection_count = len(result.boxes) if result.boxes is not None else 0
print(f"Jumlah detection: {detection_count}")
print(f"Total latency   : {wall_clock_ms:.1f} ms")
print(f"YOLO speed      : {result.speed}")

## 6. Tampilkan dan simpan visualisasi

`result.plot()` menggambar bounding box serta confidence. Ultralytics menghasilkan array BGR, sehingga warna dikonversi ke RGB hanya untuk ditampilkan oleh Matplotlib. File asli tetap disimpan sebagai JPEG BGR melalui OpenCV.

In [ ]:
annotated = result.plot()
output_image_path = OUTPUT_DIR / f"{image_path.stem}_annotated.jpg"
if not cv2.imwrite(str(output_image_path), annotated):
    raise RuntimeError(f"Gagal menyimpan hasil ke {output_image_path}")

plt.figure(figsize=(16, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f"Shelf Detection POC - {detection_count} products - {wall_clock_ms:.1f} ms")
plt.axis("off")
plt.show()

print(f"Gambar anotasi disimpan: {output_image_path}")

## 7. Simpan hasil terstruktur sebagai JSON

Setiap detection disimpan dengan koordinat `xyxy`, confidence, class ID, dan nama class. Laporan juga mencatat checksum model, parameter inferensi, ukuran gambar, serta rincian latency agar POC dapat direproduksi.

In [ ]:
detections = []
if result.boxes is not None:
    boxes_xyxy = result.boxes.xyxy.cpu().tolist()
    confidences = result.boxes.conf.cpu().tolist()
    class_ids = result.boxes.cls.cpu().tolist()
    for bbox, confidence, class_id_value in zip(boxes_xyxy, confidences, class_ids):
        class_id = int(class_id_value)
        detections.append(
            {
                "bbox_xyxy": [round(float(value), 2) for value in bbox],
                "confidence": round(float(confidence), 6),
                "class_id": class_id,
                "class_name": result.names.get(class_id, str(class_id)),
            }
        )

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_path": str(MODEL_PATH),
    "model_sha256": model_sha256,
    "image_path": str(image_path),
    "image_width": width,
    "image_height": height,
    "confidence_threshold": CONFIDENCE,
    "iou_threshold": IOU_THRESHOLD,
    "image_size": IMAGE_SIZE,
    "device": str(DEVICE),
    "wall_clock_ms": round(wall_clock_ms, 3),
    "yolo_speed_ms": {key: round(float(value), 3) for key, value in result.speed.items()},
    "detection_count": detection_count,
    "detections": detections,
}

output_json_path = OUTPUT_DIR / f"{image_path.stem}_result.json"
output_json_path.write_text(json.dumps(report, indent=2))
print(f"Laporan JSON disimpan: {output_json_path}")
print(json.dumps(report, indent=2)[:3000])

## 8. Evaluasi test set (opsional)

Bagian ini hanya berjalan jika `RUN_DATASET_EVALUATION = True`. Ultralytics mengunduh dan menyiapkan SKU-110K resmi secara otomatis bila belum tersedia, lalu `model.val()` menghitung mAP@50, mAP@50-95, precision, dan recall pada split test.

Gunakan bagian ini setelah inferensi satu gambar berhasil. Evaluasi seluruh test set dapat memakan waktu dan kuota GPU cukup besar.

In [ ]:
if RUN_DATASET_EVALUATION:
    metrics = model.val(
        data=DATASET_NAME,
        split="test",
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        project=str(OUTPUT_DIR),
        name="test_set_evaluation",
        exist_ok=True,
        verbose=True,
    )
    evaluation_metrics = {
        key: round(float(value), 6)
        for key, value in metrics.results_dict.items()
        if isinstance(value, (int, float))
    }
    evaluation_report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_sha256": model_sha256,
        "dataset": DATASET_NAME,
        "split": "test",
        "metrics": evaluation_metrics,
    }
    evaluation_output = OUTPUT_DIR / "test_set_metrics.json"
    evaluation_output.write_text(json.dumps(evaluation_report, indent=2))
    print(json.dumps(evaluation_report, indent=2))
    print(f"Metrik disimpan: {evaluation_output}")
else:
    print("Evaluasi test set dilewati. Ubah RUN_DATASET_EVALUATION=True untuk menjalankannya.")

## Interpretasi hasil POC

POC awal dianggap berhasil bila model dapat dimuat, bounding box masuk akal secara visual, hasil JSON terbentuk, dan latency tercatat. Jangan menilai model hanya dari satu gambar.

Untuk kandidat MVP, gunakan test set independen dan jadikan nilai berikut sebagai titik awal, bukan aturan mutlak:

- mAP@50 minimal 0.80.
- mAP@50-95 minimal 0.50.
- Precision minimal 0.85.
- Recall minimal 0.88.

Periksa juga foto blur, gelap, miring, produk kecil, dan rak sangat padat. Detector ini hanya mengenali keberadaan objek produk, belum mengenali identitas SKU.